# QC IV: Shor's Algorithm

We have now covered enough material to discuss the famous Shor's factoring algorithm. If you have not heard about this, this can be quickly summarized as providing a 
*probabilistic* factoring algorithm that runs in polynomial time. Recall that general factoring of integers is supposed to be a notoriously difficult comptuational task, so Shor's polynomial algorithm was surprising when it was first introduced (in 1994!). Needless to say, this spurred on a widespread interest in quantum computing (and piqued the interest of many cryptographers). 

Shor's algorithm consists of two main parts: the part that requires quantum computing, and the part that is purely classical processing. It may please you to hear that we've already covered the crux of the quantum computing portion of Shor's algorithm. The quantum part of Shor's algorithm is acutally fairly straightforward, and is a simple application of the quantum Fourier transform via phase estimation. Perhaps this is a testament to the ubiquity of the quantum Fourier transform. Most of the hardwork will revolve around extracting the necessary information out of a QPE measurement. 

At the high level, Shor's algorithm can be summarized as follows. First, the quantum part requires us to convert the task of factoring an integer $N$ to a matter of 
calculating eigenvales for some unitary operator $U$ acting on some qubit system $\mathcal{H}(n)$. To do this, we will attempt to reduce the factoring problem to the *order-finding* problem: that is, for a fixed integer $a$, calculate the smallest $r$ such that $a^{r} \equiv 1$ (mod $N$). So, the algorithm begins by first randomly selecting an integer $0 < a < N$. Then, running the QPE procedure for $U$ leaves us with a qubit system in some quantum state that we can measure to obtain some classical value $v$. The measured classical value $v$ will not directly contain the answer to our order-finding problem (i.e., the integer $r$ for the fixed integer $a$), but $v$ will be a value sufficiently close to a multiple of $\frac{2^{n}}{r}$. What will happen then is that we will apply classical methods to extract the true order $r$ from the measured value $v$. 

That sounds straightforward enough -- but alas, it turns out that the order-finding problem does not *always* give us a factor of $N$. It will do so only if $r$ happens to be <b>even</b>. Furthermore, we will shortly see that $r$ being an even integer may still fail to provide us with a factor of $N$. If either $a^{r/s} + 1$ or $a^{r/2} - 1$ happens to be a multiple of $N$, then we will not be guaranteed a factor of the integer $N$.

However, of course the order $r$ depends on the randomly chosen integer $a$ from the beginning. We can just keep repeating the above steps if necessary, until we stumble upon a choice of integer $a$ for which $r$ happens to be even, and neither  $a^{r/s} + 1$ or $a^{r/2} - 1$ happen to be multiples of $N$. As long as the steps above are polynomial in time complexity, repeating the above steps a number of times until we happen upon a good choice of integer $a$ does not ultimately detract from the overall polynomial time complexity of the algorithm. 


Now let's discuss the details.

------

### Classical reduction of factoring to order-finding


Recall that the order of an integer $a$ in $(\mathbb{Z}/ N \mathbb{Z})^{\times}$ is the smallest integer $r$ such that $a^{r} \equiv 1$ modulo $N$. The important insight is that if $r$ is even, then we can write $ (a^{r/2})^{2} \equiv 1 $  modulo $N$, or $(a^{r/2})^{2} - 1 \equiv 0  (\text{mod } N)$. This gives us

$$
( a^{r/2} + 1 )(a ^{r/2} - 1 ) \equiv 0 (\text{mod } N )
$$

Thus, as long as neither $( a^{r/2} + 1 )$ or $(a ^{r/2} - 1 )$ is a multiple of $N$, then we are guaranteed that both $( a^{r/2} + 1 )$ and $(a ^{r/2} - 1 )$ have non-trivial common factors with $N$. 

Therefore, we can extract factors of $N$ via order-finding in the following way:

1. Randomly choose an integer $a$, and determine the order $r$ of $a \in (\mathbb{Z}/N \mathbb{Z})^{\times}$.

2. If $r$ is even, then use the Euclidean algorithm to effectively compute the gcd of $a^{r/2} + 1$ (or $a^{r/2} - 1$) and $N$. 

Repeat all the above steps if necessary (for example, if the randomly chosen $a$ results in an *odd* $r$)

-----------------

###  Using QPE to (approximately) solve the order-finding problem 

To understand how we can use the quantum phase estimation algorithm to solve the order-finding problem, we need to consider a unitary transformation
with eigenvalues whose values will allow us to compute orders of elements in $\mathbb{Z}/N\mathbb{Z}$. 

The following unitary can be implemented with $\mathcal{O}(n^{3})$ gates:
$$
U_{a} \ket{x}_{2n} = \begin{cases}  \ket{ ax (\text{mod } N)} & \text{ if } 0 \leq x \leq N  \\
                                    \ket{x} & \text{ if } x > N \end{cases}
$$

There are many ways to implement the modular multiplication gate, so it's best for us to leave this as a black-box. 
In fact, by taking a slightly worse time complexity, one can implement this in a way that requires the use of less ancilla qubits. This is demonstrated in this [paper](https://arxiv.org/abs/quant-ph/0205095) by Beauregard. A qiskit implementation of Beauregard's algorithm can be found [here](https://github.com/elhyc/ShorOracle)


Then, taking powers of $U_{a}$ acts as:

$$
U^{l}_{a} \ket{x}_{2n} = \begin{cases}  \ket{ a^{l} x (\text{mod } N)} & \text{ if } 0 \leq x \leq N  \\
                                    \ket{x} & \text{ if } x > N \end{cases}
$$


After choosing your favourite (or Beauregard's construction for concreteness) implementation of these gates, we can
suppose that we can implement controlled $U_{a}^{l}$ gates as required for the QPE algorithm.
<!-- 
$$
U_{x} \ket{y} = \ket{ xy (\text{mod } N)}
$$ -->

By direct calculation, one can see that the states defined by

$$
\ket{u_{s}} := \frac{1}{\sqrt{r}} \sum\limits_{k=0}^{r-1} \exp( \frac{ - 2\pi i s k }{r} )  \ket{ a^{k}  (\text{mod } N)}
$$

for integers $0 \leq s \leq r - 1$ are eigenvectors of $U_{a}$, where $r$ is the order of $a$ in $(\mathbb{Z}/N\mathbb{Z})^{\times}$.
Indeed, we have that 

$$
U_{a} \ket{u_{s}} = \frac{1}{\sqrt{r}} \sum\limits_{k=0}^{r-1} \exp( \frac{ - 2\pi i s k }{r} ) \ket{ a^{k + 1}  (\text{mod } N)}
$$

$$
= \exp( \frac{ - 2\pi i s }{r} ) \ket{u_{s}}
$$

So that $\ket{u_{s}}$ has corresponding eigenvalue $\exp( \frac{ - 2\pi i s }{r} )$. Therefore, using the QPE algorithm, we will be able to approximate $\exp( \frac{ - 2\pi i s }{r} )$ to a desired degree of precision, and then we are left with the task of extracting the value $r$ out of our approximation. 



However, we are not out of the woods yet -- recall that to perform the QPE algorithm, we need also need to be able to prepare 
the eigenstate $\ket{u_{s}}$ to pass through the second register of the QPE circuit. Unfortunately, it seems that from our description of 
$\ket{u_{s}}$ above, in order to prepare the state $\ket{u_{s}}$ one must already know $r$! Fortunately, there is a workaround. The idea is that we are not necessarily interested in any individual eigenstate $\ket{u_{s}}$ -- any one of these eigenstates can be used to approximate $r$ via approximating $\exp( \frac{ - 2\pi i s }{r} )$. Therefore, we use the identity

$$
\frac{1}{\sqrt{r}} \sum\limits_{s = 0 }^{r - 1 } \ket{u_{s}} = \ket{1}_{N}
$$

which follows from the fact that $\sum \limits_{s = 1}^{r-1} e^{ \frac{2 \pi i s} { r}} = 1$. Then, the QPE circuit will act as follows:


$$
\text{QPE}_{U} \Big( \ket{0}_{2n} \ket{1}_{n} \Big) = \text{QPE}_{U} \Big( \ket{0}_{2n} (  \frac{1}{\sqrt{r}} \sum\limits_{s = 0 }^{r - 1 } \ket{u_{s}}  ) \Big)
$$
$$
= \frac{1}{\sqrt{r}} \sum\limits_{s = 0 } ^{r-1} \text{QPE}_{U} \Big( \ket{0}_{2n}  \ket{u_{s}} \Big) = \frac{1}{\sqrt{r}} \sum\limits_{s = 0}^{r-1} \ket{ 2^{2n} \frac{s}{r} }_{F} \ket{ u_{s} }
$$

Therefore, the probability of extracting an integer $k$ from $\text{QPE}_{U} \Big( \ket{0}_{2n} \ket{1}_{n} \Big)$ is given by 

$$
\begin{align*}
& \mathbb{P} \Big( k \mid \text{QPE}_{U} ( \ket{0}_{2n} \ket{1}_{n} ) \Big) = \frac{1}{r} \sum\limits_{s = 0}^{r-1} | \langle k \mid 2^{2n} \frac{s}{r} \rangle |^{2} \\
& = \frac{1}{r} \sum\limits_{s = 0}^{r-1} \frac{ \sin^{2}( 2 \pi ( 2^{2n} \frac{s}{r} - k )) }{ 4^{n} \sin^{2}( \frac{2\pi}{2^{2n}} ( 2^{2n} \frac{s}{r} - k) ) } 
\end{align*}
$$

Now this distribution is the average of all peaky distributions we get from the probability amplitudes of the individual Fejér states $\ket{ 2^{2n} \frac{s}{r} }_{F}$.  
The probability distribution for measurement outcomes will look something like this:

<img src="fejermixture.png" style="display: block; margin: auto; width: 30%">



The outcomes with the highest probabilities will be integers $k$ centered at the peaks in this mixture. That is, the most likely outcome from this measurement will be an integer $k$ such that for some $\frac{s}{r}$, 

$$
| \frac{k}{2^{2n}} - \frac{s}{r} | < \frac{1}{2 \cdot 2^{n} }
$$


<!-- Measuring the first $2n$ qubits of the resulting state will give us the QPE approximation of $\frac{s}{r}$ for some $s \in \{ 0, \cdots , r-1 \}$.  -->
<!-- 
####  Implementing controlled $U_{a}^{2^{j}}$ gates 

For the record, we will state the following.

The following unitary can be implemented with $\mathcal{O}(n^{3})$ gates:
$$
U_{a} \ket{x}_{2n} = \begin{cases}  \ket{ ax (\text{mod } N)} & \text{ if } 0 \leq x \leq N  \\
                                    \ket{x} & \text{ if } x > N \end{cases}
$$

There are many ways to implement the modular multiplication gate, so it's best for us to leave this as a black-box. 
In fact, by taking a slightly worse time complexity, one can implement this in a way that requires the use of less ancilla qubits. This is demonstrated in this [paper](https://arxiv.org/abs/quant-ph/0205095) by Beauregard. A qiskit implementation of Beauregard's algorithm can be found [here](https://github.com/elhyc/ShorOracle)


Then, taking powers of $U_{a}$ acts as:

$$
U^{l}_{a} \ket{x}_{2n} = \begin{cases}  \ket{ a^{l} x (\text{mod } N)} & \text{ if } 0 \leq x \leq N  \\
                                    \ket{x} & \text{ if } x > N \end{cases}
$$


After choosing your favourite (or Beauregard's construction for concreteness) implementation of these gates, we can
suppose that we can implement controlled $U_{a}^{l}$ gates as required for the QPE algorithm. -->

------------------------------


### Extracting the order $r$ from the QPE approximation

We've already demonstrated that we can apply the QPE algorithm to produce a dyadic rational $\frac{k}{2^{2n}}$ close to $\frac{s}{r}$ with high probability. However, we still need to extract the order $r$ from the approximation $\frac{k}{2^{2n}}$. For this, we have a lemma.

> Lemma: There exist a *unique* (reduced) fraction $s/r$ with $0 < s < r < N$ such that
>    $$ | \frac{k}{2^{2n}} - \frac{s}{r} | < \frac{1}{2^{n+1}} $$
> 
>
> Proof: 
>
> Recall that $n = \lceil \log_{2}( N ) \rceil$, so that $2^{2n} > N^{2}$. Therefore, we have that  
>   $$
>   | \frac{k}{2^{2n}} - \frac{s}{r} | < \frac{1}{2^{n+1}} < \frac{1}{2 N^{2}} 
>   $$
>
>   Suppose that there exists $\frac{s'}{r'}$ such that $\frac{s'}{r'} \neq \frac{s}{r}$ and $0 < s' < r' < N$. Then 
>   $$
>   | \frac{s}{r} - \frac{s'}{r'} | < \frac{1}{N^{2}}
>   $$
>
>  On the other hand, 
>  $$
>  | \frac{k}{r} - \frac{k'}{r'} | = \frac{ | k \cdot  r' - k' \cdot r | }{r \cdot r'} \geq \frac{1}{r \cdot r'}
>  $$
>
>  But as $r, r' < N$, we have that $ \frac{1}{ r \cdot r'} > \frac{1}{N^{2}}$. This contradiction shows that 
>  we can only have one such reduced fraction $\frac{s}{r}$.

The above lemma shows that as if we have succeeded in finding $k$ such that $| \frac{k}{2^{2n}} - \frac{s}{r} | < \frac{1}{2^{n+1}} $, then the fraction $\frac{s}{r}$ is uniquely determined as the only reduced fraction $\frac{s}{r}$ is within $\frac{1}{2^{n+1}}$ from $\frac{k}{2^{2n}}$. Note that if $\frac{s}{r}$ was an $n$-bit dyadic rational itself, then we must have that $k = s$ and $2^{2n} = r$. This is consistent with what the QPE algorithm would have outputted upon measurement. 

In fact, there is a polynomial algorithm for finding such $s/r$ as in the lemma, using *continued fractions*. 

If we take the continued fractions expansion of $\frac{k}{2^{2n}}$, we get a sequence of fractions (called convergents)

$$
\frac{1}{a_{1}}, \frac{1}{a_{1} + \frac{1}{a_{2}}} , \frac{1}{a_{1} + \frac{1}{a_{2} + \frac{1}{a_{3}}} }, \cdots
$$

which converges to $\frac{k}{2^{2n}}$. The claim is that 

1. There is an $\mathcal{O}(n^{3})$ algorithm to compute these convergents

2. The fraction $\frac{s}{r}$ arises as one of these convergents. 


However, there is a small hiccup in this procedure. The continued fractions algorithm actually returns the fraction $\frac{s}{r}$ as a convergent of the continued fractions sequence associated to $\frac{k}{2^{2n}}$. What we really want is $r$ itself, and not $\frac{s}{r}$. If $s$ and $r$ are not coprime, then we may obtain $\frac{s'}{r'} = \frac{s}{r}$ with $r'$ being a factor of $r$. Fortunately, there are ways around this issue. 

<u>Workaround 1</u>: recall that $s$ is obtained from a random measurement outcome from $\text{QPE}_{U_{a}} ( \ket{0}_{2n} \ket{1}_{n} ) $. It is actually quite likely that $s$ chosen from $0,..., r-1$ is coprime to $r$ (exercise: work out this probability), so we can just keep repeating the procedure until we get coprime $s$ and $r$.  

<u>Workaround 2</u>: If we run the whole procedure twice and extract two pairs $s_{1}, r_{1}$ and $s_{2},r_{2}$ with $\frac{s_{1}}{r_{1}} = \frac{s_{2}}{r_{2}}$, and if $s_{1}$ and $s_{2}$ are coprime then the desired $r$ can be found by taking $r$ to be the lowest common multiple of $r_{1}$ and $r_{2}$. One can show that there is at least a $1/4$ chance that this workaround will succeed (i.e. that $s_{1}$ and $s_{2}$ will be coprime). This success probability is independent of $n$, so we can always try again a constant number of times before we are likely to succeed.




Putting the QPE part and the continued fractions part together, we have a polynomial probabilistic algorithm, whose success rate is independent of $n$. We will refer to this as *Shor's order-finding algorithm*. As we can see, it is most of the work involved with executing "Shor's (factoring) algorithm"

----------------------------------
## Summarizing Shor's algorithm: end-to-end

<b>Input</b>: A composite number $N$, with $\lceil \log_{2}(N) \rceil = n$. 

<b>Output</b>: A non-trivial factor of $N$

<b>Time complexity</b>: $\mathcal{O}( n^{3} )$

<b>Probability of success</b>: relatively high chance of success. More importantly, it is independent of $n$ 

Without loss of generality, we can assume that $N$ is odd, otherwise we can just output $2$. 

Steps: 

1. Randomly choose $a$ in the range $1, ... , N-1$. If $gcd(a,N) \neq 1$, use the Euclidean algorithm to find a this gcd. Return as output. Else, move to step 2

2. Use Shor's order-finding algorithm to find the order $r$ of $a$ in $(\mathbb{Z}/N \mathbb{Z})^{\times}$. This algorithm will succeed with a relatively high probability, which is independent of $n$. This step can be repeated for some number of times before moving to step 3

3. If $r$ is even and $a^{r/2} \neq 1 (\text{mod} N)$, then we compute $gcd(x^{r/2}-1, N)$ and $gcd(x^{r/2}+1, N)$. If neither $gcd(x^{r/2}-1, N)$ or $gcd(x^{r/2}+1, N)$
are $1$ or $N$, then return one of them as the output, as it is a non-trivial factor of $N$. Otherwise, the algorithm fails. 

------------------------------------
The Jupyter notebook for Part IV can be found [here](https://github.com/elhyc/qcfieldscourse/blob/main/qc4.ipynb)
